<a href="https://colab.research.google.com/github/Saransh-Basu-01/Advanced-Pytorch/blob/main/Hands_on_Practical_Complete_Training_Routine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader

In [2]:
lr=0.01
epochs=100
batch_size=16

In [4]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using device:{device}")

using device:cuda


In [5]:
# Generate synthetic data: y = 2x + 1 + noise
true_weight = torch.tensor([[2.0]])
true_bias = torch.tensor([1.0])

# Generate training data
X_train_tensor = torch.randn(100, 1) * 5 # 100 examples, 1 feature
y_train_tensor = true_weight * X_train_tensor + true_bias + torch.randn(100, 1) * 0.5 # Add some nois

In [13]:
X_train_tensor[0]

tensor([-3.7058])

In [8]:
# Generate validation data (separate set)
X_val_tensor = torch.randn(20, 1) * 5 # 20 examples, 1 feature
y_val_tensor = true_weight * X_val_tensor + true_bias + torch.randn(20, 1) * 0.5


In [10]:
# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

# Create dataloaders
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False) # No need to shuffle validation data

In [12]:
train_dataset[0]

(tensor([-3.7058]), tensor([-6.1216]))

In [17]:
# Define the model (a simple linear layer)
# Input feature size = 1, Output feature size = 1
model = nn.Linear(1, 1).to(device) # Move model to the selected device

# Define the loss function (Mean Squared Error for regression)
loss_fn = nn.MSELoss()

# Define the optimizer (Stochastic Gradient Descent)
optimizer = optim.SGD(model.parameters(), lr=lr)

print("Model definition:")
print(model)
print("\nInitial parameters:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: {param.data.squeeze()}")


Model definition:
Linear(in_features=1, out_features=1, bias=True)

Initial parameters:
weight: 0.4427163600921631
bias: -0.07686054706573486


In [20]:
len(train_loader)

7

In [18]:
print("\nStarting Training...")
for epoch in range(epochs):
    model.train() # Set the model to training mode
    running_loss = 0.0
    num_batches = 0

    # Iterate over batches from the DataLoader
    for i, (features, labels) in enumerate(train_loader):
        # Move batch data to the same device as the model
        features = features.to(device)
        labels = labels.to(device)

        # 1. Forward pass: Compute model's predictions
        outputs = model(features)

        # 2. Calculate the loss
        loss = loss_fn(outputs, labels)

        # 3. Backward pass: Compute gradients
        # First, zero the gradients from the previous step
        optimizer.zero_grad()
        # Then, perform backpropagation
        loss.backward()

        # 4. Optimizer step: Update model weights
        optimizer.step()

        # Accumulate loss for reporting
        running_loss += loss.item()
        num_batches += 1

    # Print average loss for the epoch    for i, (features, labels) in enumerate(train_loader):
    avg_epoch_loss = running_loss / num_batches
    if (epoch + 1) % 10 == 0: # Print every 10 epochs
        print(f"Epoch [{epoch+1}/{epochs}], Training Loss: {avg_epoch_loss:.4f}")

print("Training Finished!")



Starting Training...
Epoch [10/100], Training Loss: 0.3329
Epoch [20/100], Training Loss: 0.2353
Epoch [30/100], Training Loss: 0.2275
Epoch [40/100], Training Loss: 0.2221
Epoch [50/100], Training Loss: 0.2396
Epoch [60/100], Training Loss: 0.2541
Epoch [70/100], Training Loss: 0.2230
Epoch [80/100], Training Loss: 0.2248
Epoch [90/100], Training Loss: 0.2135
Epoch [100/100], Training Loss: 0.2309
Training Finished!


In [21]:
print("\nStarting Evaluation...")
model.eval() # Set the model to evaluation mode
total_val_loss = 0.0
num_val_batches = 0

# Disable gradient calculations for evaluation
with torch.no_grad():
    for features, labels in val_loader:
        # Move batch data to the device
        features = features.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(features)

        # Calculate loss
        loss = loss_fn(outputs, labels)
        total_val_loss += loss.item()
        num_val_batches += 1

avg_val_loss = total_val_loss / num_val_batches
print(f"Validation Loss: {avg_val_loss:.4f}")

# Inspect the learned parameters
print("\nLearned parameters:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: {param.data.squeeze()}")

print(f"(True weight: {true_weight.item():.4f}, True bias: {true_bias.item():.4f})")




Starting Evaluation...
Validation Loss: 0.4086

Learned parameters:
weight: 1.9791395664215088
bias: 0.887770414352417
(True weight: 2.0000, True bias: 1.0000)


In [ ]:
# Saving the model's learned parameters
model_save_path = 'linear_regression_model.pth'
torch.save(model.state_dict(), model_save_path)
print(f"\nModel state_dict saved to {model_save_path}")

# Example of loading the model state
# First, instantiate the model architecture again
loaded_model = nn.Linear(1, 1).to(device)

# Then, load the saved state dictionary
loaded_model.load_state_dict(torch.load(model_save_path))
print("Model state_dict loaded successfully.")

# Remember to set the loaded model to evaluation mode if using for inference
loaded_model.eval()

# You can now use loaded_model for predictions
# Example prediction with the loaded model:
with torch.no_grad():
    sample_input = torch.tensor([[10.0]]).to(device) # Example input
    prediction = loaded_model(sample_input)
    print(f"Prediction for input 10.0: {prediction.item():.4f}")
    # Expected output should be close to 2*10 + 1 = 21
